# درس ۰۱ - مقدمه‌ای بر عوامل هوش مصنوعی

به اولین درس دوره **عوامل هوش مصنوعی برای مبتدیان** خوش آمدید!

یک **عامل هوش مصنوعی** برنامه‌ای است که از یک مدل زبان بزرگ (LLM) به عنوان موتور منطق خود استفاده می‌کند و می‌تواند در دنیای واقعی *اقداماتی* انجام دهد — فراخوانی APIها، پرس‌وجو از پایگاه‌های داده، یا اجرای کد — تا به نمایندگی از یک کاربر هدفی را محقق کند.

در این دفترچه یادداشت شما اولین عامل خود را می‌سازید: یک **عامل سفر** که مقاصد تعطیلات را پیشنهاد می‌دهد. در این مسیر شما یاد خواهید گرفت چگونه:

1. به سرویس عامل Azure AI Foundry با استفاده از **چارچوب عامل مایکروسافت** متصل شوید.
2. به عامل یک **ابزار** بدهید — یک تابع ساده پایتون که بتواند آن را فراخوانی کند.
3. عامل را اجرا کنید و پاسخ آن را بررسی کنید.
4. پاسخ عامل را به صورت توکن به توکن پخش (استریم) کنید.


## راه‌اندازی

قبل از اجرای این دفترچه، مطمئن شوید که:

1. **یک پروژه Azure AI Foundry** با یک مدل چت مستقر شده (مثلاً `gpt-4o-mini`) دارید.
2. **با Azure CLI وارد شده‌اید** — دستور `az login` را در ترمینال خود اجرا کنید.
3. **متغیرهای محیطی لازم را تنظیم کرده‌اید:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Azure AI Foundry شما.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.

سلول زیر بسته‌های پایتونی مورد نیاز شما را نصب می‌کند.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ایجاد اولین عامل شما

یک عامل به دو چیز نیاز دارد:

- **دستورات** که به آن می‌گویند *که کیست* و *چگونه رفتار کند* (یک پرامپت سیستمی).
- **ابزارها** — توابع پایتون که با `@tool` تزئین شده‌اند و عامل می‌تواند برای دریافت اطلاعات یا انجام اقدامات آنها را فراخوانی کند.

در زیر یک ابزار ساده تعریف می‌کنیم که فهرستی از مقصدهای محبوب تعطیلات را برمی‌گرداند. عامل هنگام درخواست کاربر برای توصیه‌های سفر از این ابزار استفاده خواهد کرد.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## پاسخ‌های جاری

برای تجربه‌ای تعاملی‌تر می‌توانید پاسخ عامل را به‌صورت **جاری** دریافت کنید. به جای اینکه منتظر پاسخ کامل بمانید، عامل قطعات متنی را به محض تولید ارائه می‌دهد. این ویژگی به‌ویژه در رابط‌های چت که می‌خواهید خروجی را به‌صورت زنده نمایش دهید، مفید است.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصه

در این درس یاد گرفتید چگونه:

- **یک ارائه‌دهنده ایجاد کنید** که از طریق `AzureAIProjectAgentProvider` به سرویس Azure AI Foundry Agent متصل می‌شود.
- **یک ابزار تعریف کنید** با استفاده از دکوراتور `@tool` تا عامل بتواند توابع پایتون شما را فراخوانی کند.
- **عامل را اجرا کنید** با پیام کاربر و پاسخ آن را چاپ کنید.
- **پاسخ‌ها را به صورت جریان** برای خروجی در زمان واقعی دریافت کنید.

در درس بعدی چارچوب‌های عاملی را عمیق‌تر بررسی خواهیم کرد و یاد می‌گیریم چگونه ابزارهای قدرتمندتر و قابلیت استدلال چند مرحله‌ای به عوامل بدهیم.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح‌مسئولیت**:  
این سند با استفاده از سرویس ترجمه ماشینی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
